In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

# 1. Load data
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# CRITICAL: Scale pixel values to between 0 and 1
x_train = x_train / 255.0
x_test = x_test / 255.0

# 2. Build the model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    # CRITICAL: 10 output nodes with softmax activation
    Dense(10, activation='softmax') 
])

# 3. Compile the model
model.compile(optimizer='adam',
              # CRITICAL: Must be 'sparse_' because labels are integers, not arrays
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# 4. Train the model
print("Training model...")
model.fit(x_train, y_train, epochs=5)

# 5. Verify it actually works on the test data
print("\nEvaluating on test data...")
test_loss, test_acc = model.evaluate(x_test, y_test)

# 6. Save the fixed model
model.save("mnist_model.keras")
print("Saved healthy model to mnist_model.keras")

c:\Users\HP\Desktop\Matplot\Matplot\Masai\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training model...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9241 - loss: 0.2642
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9660 - loss: 0.1160
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9768 - loss: 0.0784
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9814 - loss: 0.0593
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9861 - loss: 0.0449

Evaluating on test data...
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9760 - loss: 0.0769
Saved healthy model to mnist_model.keras


In [2]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Load the saved model
model = load_model("mnist_model.keras")


def preprocess_digit(roi):
    """
    Turn a webcam ROI into a 28x28 array that actually matches the
    distribution MNIST digits were trained on:
      - digit isolated from the background (not the whole box squashed down)
      - scaled so it fills ~20 of the 28 pixels, like real MNIST digits
      - centered on the canvas
    Returns None if nothing digit-like is found in the ROI.
    """
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu picks the threshold automatically instead of a fixed 128,
    # so it survives changes in room lighting.
    _, thresh = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Find the digit's bounding box; ignore specks of noise.
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    c = max(contours, key=cv2.contourArea)
    if cv2.contourArea(c) < 200:
        return None  # nothing meaningful drawn in the box yet

    x, y, w, h = cv2.boundingRect(c)
    digit = thresh[y:y + h, x:x + w]

    # Scale so the longer side becomes 20px (matches the original MNIST
    # normalization, where digits sit inside a 20x20 box on a 28x28 canvas).
    scale = 20.0 / max(w, h)
    new_w, new_h = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Webcam strokes are thin after resizing; thicken slightly to look
    # more like MNIST "ink".
    digit = cv2.dilate(digit, np.ones((2, 2), np.uint8), iterations=1)

    # Paste onto a blank 28x28 canvas, centered.
    canvas = np.zeros((28, 28), dtype=np.uint8)
    x_off = (28 - new_w) // 2
    y_off = (28 - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = digit

    return canvas


# 2. Start webcam stream (0 is default camera)
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Define a target box (Region of Interest) on screen
    x1, y1, x2, y2 = 200, 100, 480, 380

    # IMPORTANT: snapshot the ROI as an independent copy BEFORE drawing
    # anything on `frame`. `frame[y1:y2, x1:x2]` alone is just a *view* into
    # the same memory, so if the rectangle is drawn first (or even after,
    # since it's still the same underlying array) the green border line
    # bleeds into the crop and gets picked up as the "biggest contour" in
    # preprocess_digit, defeating the digit isolation entirely.
    roi = frame[y1:y2, x1:x2].copy()

    # Now it's safe to draw the box for the on-screen display only
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    processed = preprocess_digit(roi)

    if processed is not None:
        normalized = processed.astype("float32") / 255.0
        input_data = np.expand_dims(normalized, axis=(0, -1))

        # 3. Predict digit
        prediction = model.predict(input_data, verbose=0)
        digit = np.argmax(prediction)
        confidence = np.max(prediction)

        text = f"Digit: {digit} ({confidence * 100:.1f}%)"
        display_img = processed
    else:
        text = "No digit detected"
        display_img = np.zeros((28, 28), dtype=np.uint8)

    # Overlay result on live camera feed
    cv2.putText(frame, text, (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("MNIST Camera Feed", frame)
    cv2.imshow("Model Input (28x28)", display_img)

    # Press 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
